#### From Text to Diagnosis:
#### Evaluating GPT-Based Models for Classifying Depression Severity in Online Texts

# Script 1 - PREPROCESSING & EDA

In [ ]:
### IMPORTS ###

import os

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

import openai

import nltk
from nltk.corpus import stopwords
# If stopwords not up to date uncomment the line below:
# nltk.download('stopwords')

# Regular exressions lib.
import re

## PREPROCESSING

### Combine the Dreadit dataset into one file.

In [ ]:
# Function to append two csvs and export them as a new csv.
def combine_csvs(csv1, csv2, output):
    df1 = pd.read_csv(csv1)
    df2 = pd.read_csv(csv2)

    combined_df = pd.concat([df1, df2], ignore_index=True)
    combined_df.to_csv(output, index=False)
    print(f"Combined CSV saved.")
    
# Function to check csv length.
def check_csv_length(csv_file):
    df = pd.read_csv(csv_file)
    
    length = len(df)
    
    return length

In [ ]:
csv1 = 'C:/Users/Main/Desktop/dreaddit/dreaddit-test.csv'
csv2 = 'C:/Users/Main/Desktop/dreaddit/dreaddit-train.csv'
output = 'C:/Users/Main/Desktop/dreaddit/dreaddit.csv'

combine_csvs(csv1, csv2, output)

# Validate successful combination by checking csv length.
length1 = check_csv_length(csv1)
length2 = check_csv_length(csv2)
length3 = check_csv_length(output)

print(f"The length of csv1 is {length1} rows.")
print(f"The length of csv2 is {length2} rows.")
print(f"The length of output is {length3} rows.")
print(f"Does it add up? {length3 == length2 + length1} ({length3} == {length2} + {length1})")

### Ensure that the text in the two datasets is identical.

In [ ]:
# Function to check whether two (text) columns in two datasets are identical.
def compare_cols(csv1, col1, csv2, col2):
    df1 = pd.read_csv(csv1)
    df2 = pd.read_csv(csv2)

    # Check if passed column names exist in the dfs.
    if col1 not in df1.columns:
        return f"Column '{col1}' not found in '{csv1}'"
    if col2 not in df2.columns:
        return f"Column '{col2}' not found in '{csv2}'"

    # Store the column data.
    col1_data = df1[col1]
    col2_data = df2[col2]

    # Check if lengths are the same (already did it, but, you never know...).
    if len(col1_data) != len(col2_data):
        return "Columns are not the same length."

    # Check equality after sorting (required since using equals() function).
    are_columns_equal_sorted = col1_data.sort_values().reset_index(drop=True).equals(
        col2_data.sort_values().reset_index(drop=True)
    )

    return {
        "Are columns equal?": are_columns_equal_sorted
    }

In [ ]:
csv1 = 'C:/Users/Main/Desktop/dreaddit/dreaddit.csv'
col1 = 'text'
csv2 = 'C:/Users/Main/Desktop/Reddit_depression_dataset.csv'
col2 = 'text'

result = compare_cols(csv1, col1, csv2, col2)
print(result)

### Transfer DepSeverity rating column to Dreadit dataset.

In [ ]:
# A column (col) from csv1 is added to csv2.
def add_column_from_df(df1, key1, col, df2, key2, output_csv):
    
    # Check if passed column names exist in the dfs.
    if key1 not in df1.columns:
        return f"Column '{key1}' not found in df1"
    if col1 not in df1.columns:
        return f"Column '{col}' not found in df1"
    if key2 not in df2.columns:
        return f"Column '{key2}' not found in df2"

    # Merge the dataframes on the key.
    merged_df = df2.merge(df1[[key1, col]], how='left', left_on=key2, right_on=key1)

    # Drop the second key column if the name differs.
    # Note: Merge function retains both key cols if name is different.
    if key1 != key2:
        merged_df.drop(columns=[key1], inplace=True)

    # Save the merged dataframe to a new csv.
    merged_df.to_csv(output, index=False)
    print(f"Merged CSV saved as '{output}'")

In [ ]:
# csv from which the column is pulled.
csv1 = 'C:/Users/Main/Desktop/Reddit_depression_dataset.csv'
# Common column that acts as key.
key1 = 'text'
# Column to add.
col  = 'label'

# csv to add to column to.
csv2 = 'C:/Users/Main/Desktop/dreaddit/dreaddit.csv'
# Common column that acts as key.
key2 = 'text'

# Output csv.
output = 'C:/Users/Main/Desktop/data.csv'


df1 = pd.read_csv(csv1)
df2 = pd.read_csv(csv2)

# Rename the column in DepSeverity to avoid overwriting the label of Dreadit.
df1_renamed = df1.rename(columns={col: col + '_multi'})

# Call function.
add_column_from_df(df1_renamed, key1, col + '_multi', df2, key2, output)

### REDUNDANT: Encoding "problems".

When reading the csvs we had the problematic sequence of characters below appear when accessing the data through text editors, most likely due to encoding/decoding errors.
This led to the creation of this function to replace them according to a map, however, it appears that when reading the data through pandas the encoding errors do not appear present, therefore this part is redundant.

In [ ]:
df = pd.read_csv('C:/Users/Main/Desktop/data.csv')

# Debugging step: Print some examples to check replacements
print(df['text'].head())

In [ ]:
# Function to replace text according to dictionary.
def replace_problematic_sequences(text, replacement_map):
    # Ensure that text is a string.
    if isinstance(text, str):
        for wrong, correct in replacement_map.items():
            text = text.replace(wrong, correct)
    return text

In [ ]:
# Encoding map from: http://www.i18nqa.com/debug/utf8-debug.html
replacement_map = {
    'Ã¢â‚¬â„¢': "'",
    'Ã¢â‚¬Å“': '"',
    'Ã¢â‚¬Â': '"',
    'Ã¢â‚¬â€œ': 'â€“',
    'Ã¢â‚¬â€': 'â€”',
    'Ã¢â‚¬Ëœ': "'",
    'Ã¢â‚¬Â¢': 'â€¢',
    'Ã¢â‚¬Â¦': 'â€¦',
    'Ã¢â‚¬â€¹': '',
    'Ã¢â‚¬â€œ': "â€“",
    'Ã¢â‚¬"': '"',
}

# Load the dataset.
csv = 'C:/Users/Main/Desktop/data.csv'
df = pd.read_csv(csv)

# Replace the sequences in the 'text' column.
df['text'] = df['text'].apply(lambda x: replace_problematic_sequences(x, replacement_map))

# Print head as a check.
print(df['text'].head())

# Save the df to a new CSV file.
cleaned_file_path = 'C:/Users/Main/Desktop/data v2.csv'
df.to_csv(cleaned_file_path, index=False)

### Add columns for severity numerals and domain.

In [ ]:
# Load data.
csv = 'C:/Users/Main/Desktop/data.csv'
df = pd.read_csv(csv)

# Dictionary mapping each subreddit to a domain.
domain_mapping = {
    "domesticviolence": "abuse",
    "survivorsofabuse": "abuse",
    "anxiety": "anxiety",
    "stress": "anxiety",
    "almosthomeless": "financial",
    "assistance": "financial",
    "food_pantry": "financial",
    "homeless": "financial",
    "ptsd": "PTSD",
    "relationships": "social"
}

# Apply domain mapping to create a new col.
df['domain'] = df['subreddit'].map(domain_mapping)

# Dictionary mapping the severity levels to numerals.
severity_map = {
    'minimum': 0,
    'mild': 1,
    'moderate': 2,
    'severe': 3
}

# Apply domain mapping to create a new col.
df['multi_label_number'] = df['label_multi'].map(severity_map)
#could add .fillna(df['label_multi']) if nulls exist...

# Save the df to a new CSV file.
path = 'C:/Users/Main/Desktop/data v3.csv'
df.to_csv(path, index=False)

# Print head as a check.
print(df.head())

### Spit data in train/test/validation sets.

In [ ]:
# Load data.
csv = 'C:/Users/Main/Desktop/data v3.1.csv'
df = pd.read_csv(csv)

df['strata'] = df['multi_label_number'].astype(str) + "_" + df['domain']
# Domain vs. subreddit: subreddit cannot create a representiative set due to small size.
print(df['strata'].value_counts())

# Split data into temp (test + train) + validation.
validation_df, temp_df = train_test_split(df, test_size=0.9, stratify=df['strata'], random_state=3)

# Split temp into train and test.
train_df, test_df = train_test_split(temp_df, test_size=1/9, stratify=temp_df['strata'], random_state=3)

# Exports dfs to csvs.
validation_df.to_csv('C:/Users/Main/Desktop/data/data validation.csv', index=False)
test_df.to_csv('C:/Users/Main/Desktop/data/data test.csv', index=False)
train_df.to_csv('C:/Users/Main/Desktop/data/data train.csv', index=False)

#'C:/Users/Main/Desktop/data/validation_set.csv'

In [ ]:
# Print the distribution of sets to ensure representativeness:

# List of sets:
datasets = {
    'Train Set': train_df,
    'Test Set': test_df,
    'Validation Set': validation_df
}

# List of columns to analyze:
columns_to_analyze = ['multi_label_number', 'domain']

# Loop through each dataset to plot the distribution of the columns.
for name, df in datasets.items():
    print(f"--- {name} ---")
    
    for column in columns_to_analyze:
        distribution = df[column].value_counts()
        
        print(f"Distribution of {column} in {name}:")
        print(distribution)
        
        plt.figure(figsize=(10, 6))
        distribution.plot(kind='bar')
        plt.title(f'Distribution of {column.capitalize()} Levels in {name}')
        plt.xlabel(column.capitalize())
        plt.ylabel('Frequency')
        plt.xticks(rotation=0)
        plt.show()

### Text preprocessing

In [ ]:
############# RUN THIS TO LOAD DATA #############
df = pd.read_csv('C:/Users/scotl/OneDrive/Desktop/data/data v3.1.csv')
#################################################
# MOVE IT AROUND TO RUN A SPECIFIC PART OF CODE #
#################################################

In [ ]:
def process_text(text, remove_stopwords=False, remove_capitalization=False, remove_special_symbols=False):
    processed_text = text
    if remove_capitalization:
        processed_text = processed_text.lower()
    if remove_special_symbols:
        processed_text = re.sub(r'\W', ' ', processed_text)
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        processed_text = ' '.join([word for word in processed_text.split() if word not in stop_words])
    return processed_text

def prepare_dataframe(df):
    # Apply text transformations
    df['text_remove_stopwords'] = df['text'].apply(lambda x: process_text(x, remove_stopwords=True))
    df['text_remove_capitalization'] = df['text'].apply(lambda x: process_text(x, remove_capitalization=True))
    df['text_remove_special_symbols'] = df['text'].apply(lambda x: process_text(x, remove_special_symbols=True))
    df['text_all'] = df['text'].apply(lambda x: process_text(x, remove_stopwords=True, remove_capitalization=True, remove_special_symbols=True))

    # Combine with multi_label_number
    df_result = df[['multi_label_number', 'text_remove_stopwords', 'text_remove_capitalization', 'text_remove_special_symbols', 'text_all']]
    
    return df_result

In [ ]:
df_processed = prepare_dataframe(df)
print(df_processed.head())

### Investigation of text preprocessing in the performance of LMMs

Most of the code below is also included (and updated) in the prompt engineering script.

In [ ]:
api_key = input('Enter your OpenAI API key: ')
# Enter the OpenAI API key when prompted. Do not store API keys in notebooks.
os.environ['OPENAI_API_KEY'] = api_key
openai.api_key = os.getenv('OPENAI_API_KEY')

In [ ]:
def normalize_key_names(response_json):
    """
    Function to normalize the keys in the response JSON to ensure consistent key (and value, if text) naming.
    E.g. calculate the reponse regarless of whether the key is "depression_level" or "Depression Level".
    """
    ### Perhaps a better approach would be ignoring the key alltogether? ###
    normalized_response = {}
    for key in response_json:
        normalized_key = key.lower().replace(" ", "_")
        normalized_response[normalized_key] = response_json[key]
    return normalized_response

def compare_depression_levels(df, text_column):
    results = []

    label_mapping = {"minimum": 0, "mild": 1, "moderate": 2, "severe": 3, 0: 0, 1: 1, 2: 2, 3: 3}
    
    for i in range(len(df)):
        # Pull a row text from the data and create a prompt, based on the selected template.
        text = df.at[i, 'text']
        
        prompt_type = f"""
        Question: What is the depression level of this user?

        Output Constraint: Only return the classification of depression level (0, 1, 2, or 3) in a JSON format.

        Text data: ```{text}```
        """
        
        prompt = construct_prompt(text, prompt_type)
        
        try:
            response = get_completion(prompt, "gpt-3.5-turbo-0125")
            print(f"Row {i} - Raw response:", response)
            response = response.strip().strip('```').strip('json').strip()
            response_json = json.loads(response)
            response_json = normalize_key_names(response_json)
            
            depression_level = response_json.get('depression_level', None)

            if depression_level is None:
                print(f"Row {i} - Invalid depression level: {depression_level}")
                continue

            # Check if depression_level is already an integer or convert it if it's a string.
            # If not one of the correct values, set to -1.
            if isinstance(depression_level, str):
                depression_level = label_mapping.get(depression_level.lower(), -1)
            elif isinstance(depression_level, int):
                depression_level = label_mapping.get(depression_level, -1)
            
            # If fallback value:
            if depression_level == -1:
                print(f"Row {i} - Invalid depression level: {depression_level}")
                continue
            
            # Compare predicted and actual level, to populate match col with True/False.
            actual_label = int(df.at[i, 'multi_label_number'])
            result = {
                "index": i,
                "predicted_depression_level": depression_level,
                "actual_depression_level": actual_label,
                "match": depression_level == actual_label
            }
            results.append(result)
        
        except (json.JSONDecodeError, KeyError) as e:
            print(f"Error parsing response at row {i}: {e}")
            print(f"Response was: {response}")
            continue

    return results

# Function to calculate and print metrics
def calculate_and_print_metrics(results_df):
    # Check if the DataFrame is not empty and contains the 'match' column:
    if not results_df.empty and 'match' in results_df.columns:
        
        # Uncomment these to inspect the process:
        # COMMENT DURING FINAL RUN!
        print("Initial DataFrame:")
        print(results_df)
        
        # Remove the 'index' column if it exists
        if 'index' in results_df.columns:
            results_df = results_df.drop(columns=['index'])
        
        # Drop rows with NaN values in 'predicted_depression_level' or 'actual_depression_level'
        results_df = results_df.dropna(subset=['predicted_depression_level', 'actual_depression_level'])
        
        # Uncomment these to inspect the process:
        # COMMENT DURING FINAL RUN!
        print("After dropping NaNs:")
        print(results_df)

        # Ensure the columns are of the correct type and filter out non-numeric values.
        results_df = results_df[results_df['predicted_depression_level'].apply(lambda x: isinstance(x, int))]
        results_df['predicted_depression_level'] = results_df['predicted_depression_level'].astype(int)
        results_df['actual_depression_level'] = results_df['actual_depression_level'].astype(int)
        
        # Uncomment these to inspect the process:
        # COMMENT DURING FINAL RUN!
        print("After filtering non-numeric values:")
        print(results_df)

        # Calculate and print metrics.
        accuracy = results_df['match'].mean()
        print(f"Accuracy: {accuracy * 100:.2f}%")

        y_true = results_df['actual_depression_level']
        y_pred = results_df['predicted_depression_level']

        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        conf_matrix = confusion_matrix(y_true, y_pred)

        print(f"Mean Absolute Error (MAE): {mae}")
        print(f"Root Mean Squared Error (RMSE): {rmse}")
        print("Confusion Matrix:")
        print(conf_matrix)
    else:
        print("No valid results to display or 'match' column missing.")

In [ ]:
for column in ['text_remove_stopwords', 'text_remove_capitalization', 'text_remove_special_symbols', 'text_all']:
    print(f"Processing with column: {column}")
    comparison_results = compare_depression_levels(processed_df, column)
    results_df = pd.DataFrame(comparison_results)
    calculate_and_print_metrics(results_df)

For the results of the preprocessing, see attached file.

In [ ]:
# Final sanity check to ensure the lack of null values.

csv = 'C:/Users/Main/Desktop/data v3.1.csv'
df = pd.read_csv(csv)
print(df.isna().sum())

print(df['subreddit'].isna().sum())

# EDA

Kindly note that the EDA process was also conducted outside of Python;
primarily visualisations of LIWC metrics with PowerBI & some additinal text exploration using Power Query.

In [ ]:
# Load the CSV file
file_path = 'C:/Users/Main/Desktop/data v3.1.csv'
data = pd.read_csv(file_path)

# Visualize the distribution of 'label_multi' by subreddit
plt.figure(figsize=(14, 8))
data.groupby('subreddit')['label_multi'].value_counts().unstack().plot(kind='bar', stacked=True)
plt.title('Distribution of label_multi by Subreddit')
plt.xlabel('Subreddit')
plt.ylabel('Count')
plt.legend(title='label_multi')
plt.show()

# Calculate the average number of characters in the 'text' column
average_chars = data['text'].apply(len).mean()

# Calculate the average number of characters in the 'text' column by subreddit
average_chars_by_subreddit = data.groupby('subreddit')['text'].apply(lambda x: x.str.len().mean())

print("Average number of characters in the 'text' column:", average_chars)
print("\nAverage number of characters in the 'text' column by subreddit:")
print(average_chars_by_subreddit)

In [ ]:
# Calculate the value counts.
value_counts = data['label_multi'].value_counts()

# Print the value counts.
print(value_counts)

In [ ]:
# Calculate average characters.
average_chars = data['text'].apply(len).mean()
average_chars

Inspiration for the design for the following plot was from here:
https://python-graph-gallery.com/basic-barplot-with-seaborn/

Although, the code uploaded there does not match the results!
Quite a few tweaks needed...

In [ ]:
# Load the CSV file
# C:/Users/scotl/OneDrive/Desktop/data
#'C:/Users/Main/Desktop/data v3.1.csv'
file_path = 'C:/Users/scotl/OneDrive/Desktop/data/data v3.1.csv'
data = pd.read_csv(file_path)

# Manually reorder the DataFrame based dep sev order.
desired_order = ['minimum', 'mild', 'moderate', 'severe']
label_counts['label_multi'] = pd.Categorical(label_counts['label_multi'], categories=desired_order, ordered=True)
label_counts = label_counts.sort_values('label_multi')

# Set the figure size
plt.figure(figsize=(14, 10))

# Set the background color for the figure to a lighter gray
plt.gcf().set_facecolor('whitesmoke')

# Plot bar chart
ax = sns.barplot(
    x="label_multi",
    y="count",
    data=label_counts,
    ci=None,
    color='#69b3a2')

'''
Finalise parameters:
axes background color, grid lines along the y-axis (set to the back)
title and labels with increased font size and labelpad for distance & tick label size
'''
ax.set_facecolor('gainsboro')


ax.yaxis.grid(True, color='white', linestyle='-', linewidth=1.5)
ax.xaxis.grid(False)  # Disable the grid for the x-axis

ax.set_axisbelow(True)

plt.xlabel('Depression Severity Levels', fontsize=16, labelpad=20)
plt.ylabel('Count', fontsize=16, labelpad=20)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.show()

### LIWC & Readability by class

In [ ]:
metrics_columns = [
    'lex_liwc_i',
    'lex_liwc_posemo',
    'lex_liwc_negemo',
    'lex_liwc_anx',
    'lex_liwc_social',
    'syntax_fk_grade',
    'syntax_ari'
]

# Calculate averages grouped by 'label_multi'
grouped_averages = data.groupby('label_multi')[metrics_columns].mean()

# Calculate the global average for these metrics
global_averages = data[metrics_columns].mean()
global_averages = pd.DataFrame(global_averages).T  # Transpose to match the grouped format
global_averages.index = ['Global Average']

# Combine grouped averages and global averages into a single DataFrame
combined_results = pd.concat([grouped_averages, global_averages])

# Specify the desired order of the rows
desired_order = ['minimum', 'mild', 'moderate', 'severe', 'Global Average']

# Reorder the DataFrame according to the specified order
ordered_results = combined_results.loc[desired_order]

# Display the results
print(ordered_results)
